In [13]:
import catboost as cb
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# --- 1. Load Data ---
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

y_train = train_df['Survived'].astype(int)
df = pd.concat(
    [train_df.assign(is_train=1), test_df.assign(is_train=0, Survived=np.nan)],
    sort=False,
).reset_index(drop=True)

# --- 2. Clean Feature Extraction ---
df['Embarked'] = df['Embarked'].fillna('S')
df['Fare'] = df.groupby('Pclass')['Fare'].transform(
    lambda x: x.fillna(x.median())
)

df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Mlle': 'Miss',
    'Mme': 'Mrs',
    'Ms': 'Miss',
}
df['Title'] = df['Title'].map(title_mapping).fillna('Rare')
df['Age'] = df.groupby(['Pclass', 'Title'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

df['IsWomanOrChild'] = (
    (df['Sex'] == 'female') | (df['Title'] == 'Master')
).astype(int)
df['Surname'] = df['Name'].apply(lambda x: x.split(',')[0].strip())
df['FamilyID'] = df['Surname'] + '_' + df['Fare'].round(2).astype(str)

# --- 3. Build Pure Group Survival Maps ---
ticket_wcg = {}
family_wcg = {}

for ticket, group in df.groupby('Ticket'):
  if len(group) > 1:
    train_wc = group[
        (group['IsWomanOrChild'] == 1) & (group['Survived'].notnull())
    ]
    if len(train_wc) > 0:
      ticket_wcg[ticket] = train_wc['Survived'].mean()

for fid, group in df.groupby('FamilyID'):
  if len(group) > 1:
    train_wc = group[
        (group['IsWomanOrChild'] == 1) & (group['Survived'].notnull())
    ]
    if len(train_wc) > 0:
      family_wcg[fid] = train_wc['Survived'].mean()

# --- 4. Matrix Encoding & Model Training ---
df_encoded = pd.get_dummies(
    df, columns=['Sex', 'Embarked', 'Title', 'Pclass'], drop_first=True
)
cols_to_drop = [
    'PassengerId',
    'Name',
    'Ticket',
    'Cabin',
    'Surname',
    'FamilyID',
    'is_train',
    'Survived',
]
feature_cols = [c for c in df_encoded.columns if c not in cols_to_drop]

X_train = df_encoded[df['is_train'] == 1][feature_cols].reset_index(drop=True)
X_test = df_encoded[df['is_train'] == 0][feature_cols].reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_train), 3))
test_preds = np.zeros((len(X_test), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
  X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
  X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

  gb = GradientBoostingClassifier(
      n_estimators=100, learning_rate=0.03, max_depth=3, random_state=42
  ).fit(X_tr, y_tr)
  cb_model = cb.CatBoostClassifier(
      iterations=150, depth=3, learning_rate=0.03, verbose=0, random_seed=42
  ).fit(X_tr, y_tr)
  rf = RandomForestClassifier(
      n_estimators=200, max_depth=3, min_samples_leaf=2, random_state=42
  ).fit(X_tr, y_tr)

  oof_preds[val_idx, 0] = gb.predict_proba(X_va)[:, 1]
  oof_preds[val_idx, 1] = cb_model.predict_proba(X_va)[:, 1]
  oof_preds[val_idx, 2] = rf.predict_proba(X_va)[:, 1]

  test_preds[:, 0] += gb.predict_proba(X_test)[:, 1] / 5.0
  test_preds[:, 1] += cb_model.predict_proba(X_test)[:, 1] / 5.0
  test_preds[:, 2] += rf.predict_proba(X_test)[:, 1] / 5.0

meta_model = LogisticRegression(C=0.5, random_state=42).fit(oof_preds, y_train)
raw_test_preds = meta_model.predict(test_preds)

# --- 5. Apply Post-Processing WCG Rules (Women/Children Perishing Groups Only) ---
test_df_out = df[df['is_train'] == 0].copy().reset_index(drop=True)
test_df_out['Pred'] = raw_test_preds

overrides = 0
for i, row in test_df_out.iterrows():
  signal = ticket_wcg.get(
      row['Ticket'], family_wcg.get(row['FamilyID'], None)
  )

  # Strictly override ONLY women and children in 100% perishing groups
  if row['IsWomanOrChild'] == 1 and signal == 0.0:
    if test_df_out.loc[i, 'Pred'] != 0:
      test_df_out.loc[i, 'Pred'] = 0
      overrides += 1

print(
    f'Applied {overrides} surgical perishing overrides to ensemble'
    ' predictions.'
)

# --- 6. Export Submission ---
submission = pd.DataFrame({
    'PassengerId': test_df_out['PassengerId'].astype(int),
    'Survived': test_df_out['Pred'].astype(int),
})

assert len(submission) == 418
assert list(submission.columns) == ['PassengerId', 'Survived']
assert submission.isnull().sum().sum() == 0

submission.to_csv('submission_hybrid_wcg.csv', index=False)
print('Saved submission_hybrid_wcg.csv successfully!')

Applied 7 surgical perishing overrides to ensemble predictions.
Saved submission_hybrid_wcg.csv successfully!
